# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MisbahSangi/flyrank-ml-internship-misbah/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Setup Cell


In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MisbahSangi/flyrank-ml-internship-misbah"
REPO_DIR = "flyrank-ml-internship-misbah"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('my key removed to save it on github')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':   f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':   f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':    f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_sample':   f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected. Tables ready:", list(TABLES.keys()))

# 1. Unit of analysis + time window

**Unit of analysis:** one row = one content item (`content_hash_id`) belonging to
one client (`client_hash_id`), scored at one anchor month within the panel.

**Time window:** a 30-day **feature window** ending at the anchor date, and a
separate, non-overlapping 30-day **outcome window** immediately after it —
matching my Lane 2 (Refresh/Content Opportunity Scoring) framing from Week 1-2:
predict decline over the *next* 30 days using only what was knowable in the
*prior* 30 days.

**Anchor month for development:** `2026-03` (a mid-panel month, per the
notebook's own warning). The final month of the panel is held out, untouched,
as a sealed test month for later.

In [2]:
# Verify: confirm the grain — one row per (client, content) at the chosen anchor month
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, COUNT(*) AS n_rows
    FROM {TABLES['fact_daily']}
    WHERE date_trunc('month', report_date) = DATE '2026-03-01'
    GROUP BY 1, 2
    HAVING COUNT(*) > 1
""").df()

print(f"Rows with more than one entry per (client, content) in 2026-03: {len(grain_check)}")
print("(A well-formed daily fact table should have ONE row per day per content item —")
print(" my aggregated 'unit of analysis' is built by summing/aggregating across days,")
print(" not assuming the raw table itself is already at that grain.)")
grain_check.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with more than one entry per (client, content) in 2026-03: 331230
(A well-formed daily fact table should have ONE row per day per content item —
 my aggregated 'unit of analysis' is built by summing/aggregating across days,
 not assuming the raw table itself is already at that grain.)


,client_hash_id,content_hash_id,n_rows
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,31
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,31
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,31
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,31
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Why |
|---|---|---|
| **Feature** (known *before* the decision point, from the prior-30-day window) | `imp_prev30`, `clk_prev30`, `pos_prev30` (avg position), `visible_queries`, `rare_share`, `anon_share`, `top_query_share` | All computed strictly from data dated *before* the anchor date |
| **Label** | `is_declining` = impressions in the outcome window fell >20% vs. the feature window | The thing I'm predicting — comes from the *outcome* window only |
| **Context** (kept for grouping/reporting, never fed to the model) | `client_hash_id`, `content_hash_id`, `access_profile`, anchor month | Identifies *who/what* a row is about, not a signal about decline itself |
| **Excluded** | `imp_last30` / `clk_last30` as a *feature* | **Why excluded:** these live inside the same outcome window the label is computed from — using them as input would mean feeding the model a fragment of the answer, i.e. leakage |
| **Excluded** | `gsc_data_start` / `ga4_data_start` (raw onboarding dates) | **Why excluded:** useful only for checking whether a client has *enough history* to include at all — not a signal about whether a specific page is declining |
| **Excluded** | `trend_direction` (from the small starter CSV) | **Why excluded:** this is the exact proxy label the Week 1-2 notebooks warned about re-using as a feature — it's derived from the same outcome the warehouse label now measures directly |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# Query 1 — availability: which clients have enough history to be eligible for this anchor month
clients_check = con.sql(f"""
    SELECT
        client_hash_id,
        (gsc_data_start <= DATE '2026-03-01' - INTERVAL 60 DAY) AS has_enough_history
    FROM {TABLES['dim_clients']}
""").df()

eligible = con.sql(f"""
    SELECT COUNT(*) AS eligible_clients
    FROM ({clients_check.to_sql if False else "clients_check"})
""") if False else None

n_eligible = (clients_check['has_enough_history'] == True).sum()
print(f"Total clients: {len(clients_check)}")
print(f"Clients with >=60 days of history before anchor month (IS TRUE): {n_eligible}")
clients_check.head()

Total clients: 104
Clients with >=60 days of history before anchor month (IS TRUE): 40


,client_hash_id,has_enough_history
0,client_04660893ae39614a,<NA>
1,client_05475c07ed21a83a,<NA>
2,client_06d356715a8ff3b6,False
3,client_0797ff3a1fc9a6a5,True
4,client_08a6a72ff48e62c0,True


In [4]:
# Query 2 — counts and date span for the anchor month slice
span_check = con.sql(f"""
    SELECT
        MIN(report_date) AS first_day,
        MAX(report_date) AS last_day,
        COUNT(*) AS n_rows,
        COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {TABLES['fact_daily']}
    WHERE date_trunc('month', report_date) = DATE '2026-03-01'
""").df()
print(span_check.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 first_day   last_day  n_rows  n_content_items
2026-03-01 2026-03-31 9841378           331437


In [5]:
# Query 3 — missing values check on the key feature columns
missing_check = con.sql(f"""
    SELECT
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_impressions,
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS missing_position,
        COUNT(*) AS total_rows
    FROM {TABLES['fact_daily']}
    WHERE date_trunc('month', report_date) = DATE '2026-03-01'
""").df()
print(missing_check.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 missing_impressions  missing_position  total_rows
                 0.0         6230317.0     9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never tell you:**
- **Unbalanced history** — clients don't all have the same amount of history
  (`dim_clients.gsc_data_start` varies by months); comparing a brand-new client
  to one with a year of data isn't apples-to-apples.
- **GSC-only early rows** — some clients are `access_profile = 'gsc_only'`
  (no GA4), so engagement-style signals simply don't exist for them at all —
  not missing-at-random, structurally absent.
- **Window overlaps** — if my anchor date is chosen carelessly, the "prior 30
  days" and "next 30 days" windows could overlap at the edges; I must always
  verify `feature_window.end < outcome_window.start` for every row before
  trusting a label.
- **Causal blindness** — this data can show that a page's traffic changed; it
  cannot tell me *why* (a real algorithm update? a competitor? seasonal drift?).
  Nothing here supports a causal claim, only an associative one.

In [6]:
# Five-feature frame — each with an "available when?" line
features_frame = """
1. imp_prev30       — available when: end of the prior-30-day window (before anchor date)
2. pos_prev30       — available when: same as above, avg position over prior 30 days
3. visible_queries  — available when: query-level table is refreshed (~daily lag, before anchor)
4. top_query_share  — available when: same query-level refresh cycle as #3
5. rare_share       — available when: same query-level refresh cycle as #3
"""
print(features_frame)


1. imp_prev30       — available when: end of the prior-30-day window (before anchor date)
2. pos_prev30       — available when: same as above, avg position over prior 30 days
3. visible_queries  — available when: query-level table is refreshed (~daily lag, before anchor)
4. top_query_share  — available when: same query-level refresh cycle as #3
5. rare_share       — available when: same query-level refresh cycle as #3



In [10]:
# Build model_data: features (prior 30 days) + label (outcome 30 days), anchor = 2026-03-01
ANCHOR = "DATE '2026-03-01'"

features_and_label = con.sql(f"""
    WITH windowed AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date >= {ANCHOR} - INTERVAL 30 DAY
                      AND report_date < {ANCHOR} THEN gsc_impressions ELSE 0 END) AS imp_prev30,
            AVG(CASE WHEN report_date >= {ANCHOR} - INTERVAL 30 DAY
                      AND report_date < {ANCHOR} THEN gsc_avg_position END)      AS pos_prev30,
            SUM(CASE WHEN report_date >= {ANCHOR}
                      AND report_date < {ANCHOR} + INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_outcome30
        FROM {TABLES['fact_daily']}
        WHERE report_date >= {ANCHOR} - INTERVAL 30 DAY
          AND report_date <  {ANCHOR} + INTERVAL 30 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT *,
        (imp_outcome30 < 0.8 * imp_prev30)::INT AS is_declining
    FROM windowed
""").df()

print(f"{len(features_and_label):,} rows with enough prior-30-day history at anchor {ANCHOR}")
features_and_label.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

81,521 rows with enough prior-30-day history at anchor DATE '2026-03-01'


,client_hash_id,content_hash_id,imp_prev30,pos_prev30,imp_outcome30,is_declining
0,client_73cda7b4e4f265ea,content_078903d14ec448ed,1080.0,5.935171,1328.0,0
1,client_73cda7b4e4f265ea,content_bb09f7e0f8e1f667,1568.0,8.439471,1811.0,0
2,client_73cda7b4e4f265ea,content_47524bf5b9635e9f,3507.0,3.460354,2457.0,1
3,client_73cda7b4e4f265ea,content_78a246ede47da40b,481.0,9.366315,568.0,0
4,client_73cda7b4e4f265ea,content_7c31205c868ea7ab,134.0,5.138150,51.0,1


In [11]:
# Add query-level signals (visible_queries, top_query_share, rare_share), same pattern as notebook 03
qsignals = con.sql(f"""
    SELECT content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_queries,
        ANY_VALUE(rare_impressions_share)      AS rare_share,
        MAX(impressions_90d)                   AS top_query_impressions,
        SUM(impressions_90d)                   AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()
qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

model_data = features_and_label.merge(
    qsignals[['content_hash_id', 'visible_queries', 'top_query_share', 'rare_share']],
    on='content_hash_id', how='left'
)

# Drop rows missing any of the 5 honest features before modeling
feature_cols = ['imp_prev30', 'pos_prev30', 'visible_queries', 'top_query_share', 'rare_share']
model_data = model_data.dropna(subset=feature_cols)

print(f"Final model_data: {len(model_data):,} rows")
model_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final model_data: 66,452 rows


,client_hash_id,content_hash_id,imp_prev30,pos_prev30,imp_outcome30,is_declining,visible_queries,top_query_share,rare_share
0,client_73cda7b4e4f265ea,content_078903d14ec448ed,1080.0,5.935171,1328.0,0,10.0,0.670194,0.154891
1,client_73cda7b4e4f265ea,content_bb09f7e0f8e1f667,1568.0,8.439471,1811.0,0,23.0,0.319347,0.085102
2,client_73cda7b4e4f265ea,content_47524bf5b9635e9f,3507.0,3.460354,2457.0,1,18.0,0.211950,0.043336
3,client_73cda7b4e4f265ea,content_78a246ede47da40b,481.0,9.366315,568.0,0,17.0,0.099678,0.147604
4,client_73cda7b4e4f265ea,content_7c31205c868ea7ab,134.0,5.138150,51.0,1,6.0,0.492063,0.110687


In [13]:
# THE TRAP — deliberately add a label-derived feature, watch the score jump, then remove it
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# (assume `model_data` built from the 5 honest features + is_declining label, as in notebook 03)

# Honest version
X_honest = model_data[['imp_prev30','pos_prev30','visible_queries','top_query_share','rare_share']]
y = model_data['is_declining']
Xh_tr, Xh_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
honest_model = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xh_tr, y_tr)
print("=== Honest features only ===")
print(classification_report(y_te, honest_model.predict(Xh_te), digits=3))

# LEAKY version — sneak in a fragment of the outcome window on purpose
model_data['leaky_feature'] = model_data['imp_outcome30']  # lives INSIDE the outcome window
X_leaky = model_data[['imp_prev30','pos_prev30','visible_queries','top_query_share','rare_share','leaky_feature']]
Xl_tr, Xl_te, _, _ = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xl_tr, y_tr)
print("\n=== With leaky feature added (watch this jump) ===")
print(classification_report(y_te, leaky_model.predict(Xl_te), digits=3))

# Remove it — keep the honest number
model_data = model_data.drop(columns=['leaky_feature'])
print("\nLeaky feature removed. Honest score above is the real one to report.")

=== Honest features only ===
              precision    recall  f1-score   support

           0      0.817     0.962     0.884     13316
           1      0.458     0.129     0.201      3297

    accuracy                          0.797     16613
   macro avg      0.637     0.546     0.542     16613
weighted avg      0.746     0.797     0.748     16613


=== With leaky feature added (watch this jump) ===
              precision    recall  f1-score   support

           0      0.980     0.998     0.989     13316
           1      0.990     0.919     0.953      3297

    accuracy                          0.982     16613
   macro avg      0.985     0.958     0.971     16613
weighted avg      0.982     0.982     0.982     16613


Leaky feature removed. Honest score above is the real one to report.


**Observed:** adding `imp_outcome30` (a value used directly inside the label's
own formula) as a feature caused F1 for the declining class to jump from 0.201
to 0.953, and recall from 0.129 to 0.919 — an artificial, meaningless result,
since the model was effectively given the answer to reverse-engineer rather
than a genuine predictive signal. The honest number — 0.201 F1, 0.129 recall
on the declining class, using only features knowable *before* the outcome
window — is the real, reportable result of this lane's difficulty: predicting
*future* decline from *past* signals alone is a substantially harder problem
than detecting decline within the same window it's measured in, and this
5-feature set currently misses roughly 87% of genuine declining pages. That
low recall is itself a useful, honest finding — it points directly at where
the next modeling weeks need to focus (more features, more history, or
accepting that recall may matter more than precision for this decision).

## Self-check

Before you submit, confirm each line honestly:

- ✔ Every section above is filled — markdown thinking AND the code that backs it
- ✔ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✔ No client names, URLs, or private queries anywhere
- ✔ My claims use careful words: observed, measured, directional, decision-support
- ✔ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.